In [ ]:


import os
import sys
import argparse
import glob

import cv2
import numpy as np
import pandas as pd

# Make sure 'code/' is on the import path
sys.path.append(os.path.join(os.path.dirname(__file__), "code"))

from kalman_filter import PixelKalmanFilter
from contrast_enhancement import adaptive_gain_kalman
from roi_tracking import RoiTracker
from bmd_extraction import extract_bmd_metrics
from evaluation import evaluate_quality
from utils import ensure_dir

def process_pair(pid, raw_dir, out_dir, kf):
    """Process one HE/LE pair end-to-end, returns a dict of metrics."""
    he_path = os.path.join(raw_dir, f"{pid}_HE.png")
    le_path = os.path.join(raw_dir, f"{pid}_LE.png")
    if not os.path.exists(he_path) or not os.path.exists(le_path):
        print(f"[WARN] Missing files for {pid}, skipping.")
        return None

    # Load
    he = cv2.imread(he_path, cv2.IMREAD_GRAYSCALE)
    le = cv2.imread(le_path, cv2.IMREAD_GRAYSCALE)

    # 1. Denoise
    he_dn = kf.filter_frame(he)
    le_dn = kf.filter_frame(le)
    cv2.imwrite(os.path.join(out_dir, f"{pid}_HE_denoised.png"), he_dn)
    cv2.imwrite(os.path.join(out_dir, f"{pid}_LE_denoised.png"), le_dn)

    # 2. Contrast enhance
    he_ce = adaptive_gain_kalman(he_dn)
    le_ce = adaptive_gain_kalman(le_dn)
    cv2.imwrite(os.path.join(out_dir, f"{pid}_HE_enhanced.png"), he_ce)
    cv2.imwrite(os.path.join(out_dir, f"{pid}_LE_enhanced.png"), le_ce)

    # 3. ROI detection/tracking on HE
    tracker = RoiTracker()
    roi_mask = tracker.track(he_ce)  # boolean mask
    mask_img = (roi_mask.astype(np.uint8) * 255)
    cv2.imwrite(os.path.join(out_dir, f"{pid}_ROI.png"), mask_img)

    # 4. BMD extraction on LE enhanced
    bmd = extract_bmd_metrics(le_ce, roi_mask)

    # 5. Quality metrics on HE
    quality = evaluate_quality(he, he_dn, roi_mask)

    # Pack results
    return {"patient": pid, **bmd, **quality}

def main():
    p = argparse.ArgumentParser(
        description="DXA Kalman-preprocessing & analysis pipeline"
    )
    p.add_argument("--raw_dir",  required=True, help="Folder with PacienteXX_HE.png & _LE.png")
    p.add_argument("--out_dir",  required=True, help="Where to write images & summary")
    p.add_argument("--patients", nargs="+",
                   help="List of IDs (e.g. Paciente01). If omitted, auto-discovers from raw_dir.")
    p.add_argument("--process_var", type=float, default=1e-2, help="Kalman process variance")
    p.add_argument("--meas_var",    type=float, default=1e-1, help="Kalman measurement variance")
    args = p.parse_args()

    ensure_dir(args.out_dir)
    kf = PixelKalmanFilter(process_var=args.process_var, meas_var=args.meas_var)

    # Discover patients if not provided
    if not args.patients:
        he_files = glob.glob(os.path.join(args.raw_dir, "*_HE.png"))
        args.patients = sorted({os.path.basename(f).split("_")[0] for f in he_files})

    results = []
    for pid in args.patients:
        print(f"[INFO] Processing {pid}...")
        res = process_pair(pid, args.raw_dir, args.out_dir, kf)
        if res:
            results.append(res)

    if results:
        df = pd.DataFrame(results)
        csv_path = os.path.join(args.out_dir, "summary_metrics.csv")
        df.to_csv(csv_path, index=False)
        print(f"[INFO] Done! Summary saved to {csv_path}")
    else:
        print("[WARN] No valid patient data processed.")

if __name__ == "__main__":
    main()
